In [1]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

## Import libaries
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
from sklearn.preprocessing import StandardScaler

## Setting
base_path = '/content/drive/MyDrive/CS/'

## **데이터 불러오기**
file_path = os.path.join(base_path, 'myData')
firm1000 = pd.read_csv(os.path.join(file_path, 'firm_top1000.csv'))

firm1000['date'] = pd.to_datetime(firm1000['date'])
firm1000['year'] = firm1000['date'].dt.year

# 시드 설정
def set_seed(val):
    torch.manual_seed(val)
    torch.cuda.manual_seed(val)
    np.random.seed(val)
    random.seed(val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 메타 데이터 컬럼 정의 (결과에 포함할 컬럼들)
meta_cols = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'year', 'ret', 'date']
# 학습에 사용하지 않을 컬럼 제외
exclude_cols = meta_cols # date는 meta_cols에 포함됨
feature_cols = [c for c in firm1000.columns if c not in exclude_cols]

print(f"Feature 개수: {len(feature_cols)}")

## **CNN Projection Encoder 정의**
class MultiHeadCNNProjection(nn.Module):
    def __init__(self, input_dim, output_dims=[5, 10, 20, 50]):
        super(MultiHeadCNNProjection, self).__init__()
        self.output_dims = output_dims
        self.heads = nn.ModuleList()

        for out_dim in output_dims:
            # 단순 Projection이므로 Kernel Size=1 사용
            # 구조: Input -> Conv(Proj) -> BN -> LeakyReLU -> Conv(Proj) -> Output
            head = nn.Sequential(
                nn.Conv1d(input_dim, out_dim * 2, kernel_size=1),
                nn.BatchNorm1d(out_dim * 2),
                nn.LeakyReLU(0.1),
                nn.Conv1d(out_dim * 2, out_dim, kernel_size=1),
                nn.BatchNorm1d(out_dim),
                nn.LeakyReLU(0.1),
                # AdaptiveMaxPool1d 제거 (Sequence Length가 1이므로 불필요)
                nn.Flatten()
            )
            self.heads.append(head)

    def forward(self, x):
        # x shape: (Batch, Features, 1)
        results = {}
        for i, head in enumerate(self.heads):
            results[f'hidden_{self.output_dims[i]}'] = head(x)
        return results

# **데이터 분할 및 Hidden State 추출 (단순 Projection)**

split_year = [
    (2014, 2015, 2017, 2018),
    (2015, 2016, 2018, 2019),
    (2016, 2017, 2019, 2020),
    (2017, 2018, 2020, 2021)
]

output_dims = [5, 10, 20, 50]
SAVE_DIR = os.path.join(base_path, 'hidden_states', 'CNN_Firm_simple')

for i, (train_end, valid_start, valid_end, test_year) in enumerate(split_year):
    print(f"\n======== Round {i+1}: Test Year {test_year} ========")
    print(f"Train: ~ {train_end} | Valid: {valid_start} ~ {valid_end} | Test: {test_year}")

    # 1. 데이터 분할
    # Sliding Window가 아니므로 단순히 연도 기준으로만 자르면 됩니다.
    train_df = firm1000[(firm1000['year'] >= 1997) & (firm1000['year'] <= train_end)].copy()
    valid_df = firm1000[(firm1000['year'] >= valid_start) & (firm1000['year'] <= valid_end)].copy()
    test_df  = firm1000[firm1000['year'] == test_year].copy()

    # 2. 스케일링 (Feature만)
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])

    train_scaled = scaler.transform(train_df[feature_cols])
    valid_scaled = scaler.transform(valid_df[feature_cols])
    test_scaled  = scaler.transform(test_df[feature_cols])

    # 3. 텐서 변환
    # Conv1d 입력 형태: (Batch Size, Input Channels, Sequence Length)
    # 여기서는 Sequence Length = 1 로 설정하여 단순 Projection 수행

    # (N, Features) -> (N, Features, 1)
    tensors = {
        'train': torch.FloatTensor(train_scaled).unsqueeze(2),
        'valid': torch.FloatTensor(valid_scaled).unsqueeze(2),
        'test':  torch.FloatTensor(test_scaled).unsqueeze(2)
    }

    # 메타 데이터 (데이터 개수 변화 없음)
    meta_dict = {
        'train': train_df[meta_cols].reset_index(drop=True),
        'valid': valid_df[meta_cols].reset_index(drop=True),
        'test':  test_df[meta_cols].reset_index(drop=True)
    }

    # 4. 모델 실행 및 저장
    # 매 라운드마다 모델 초기화 (Random Projection)
    model = MultiHeadCNNProjection(len(feature_cols), output_dims)
    model.eval()

    round_dir = os.path.join(SAVE_DIR, f'Test_{test_year}')
    os.makedirs(round_dir, exist_ok=True)

    with torch.no_grad():
        for split in ['train', 'valid', 'test']:
            if tensors[split].size(0) == 0:
                continue

            features = model(tensors[split])

            for dim in output_dims:
                feats = features[f'hidden_{dim}'].numpy()
                cols = [f'hs_{dim}_{k}' for k in range(dim)]

                # Hidden States 데이터프레임
                temp_df = pd.DataFrame(feats, columns=cols)

                # 메타 데이터 붙이기 (1:1 매칭)
                current_meta = meta_dict[split]
                temp_df = pd.concat([current_meta, temp_df], axis=1)

                file_name = f'hidden_states_{split}_{dim}_Firm.csv'
                save_path = os.path.join(round_dir, file_name)

                temp_df.to_csv(save_path, index=False)

    print(f"  -> {test_year}년도 데이터 저장 완료")

print("\n 완료")

Mounted at /content/drive
Feature 개수: 51

======== Round 1: Test Year 2018 ========
Train: ~ 2014 | Valid: 2015 ~ 2017 | Test: 2018
  -> 2018년도 데이터 저장 완료

======== Round 2: Test Year 2019 ========
Train: ~ 2015 | Valid: 2016 ~ 2018 | Test: 2019
  -> 2019년도 데이터 저장 완료

======== Round 3: Test Year 2020 ========
Train: ~ 2016 | Valid: 2017 ~ 2019 | Test: 2020
  -> 2020년도 데이터 저장 완료

======== Round 4: Test Year 2021 ========
Train: ~ 2017 | Valid: 2018 ~ 2020 | Test: 2021
  -> 2021년도 데이터 저장 완료

 완료
